# Benchmark results
Each dataset has a main table and a separate spatial voxel comparison. Chamfer Full is the only Chamfer variant shown. Toggle columns without rerunning the metrics.

Each number is a sequence-weighted mean. Within a column, methods use the same sequences. The voxel table also matches sequences across available scales for each metric and polarity. Headers show the number of shared sequences. Only finite, completed scores are included. Fano scores with partial spatial support are included and flagged in the report inventory.

Blue marks the four best rank levels (darkest first, equal scores share a rank). For descriptive columns (including Same-ts), cells show the mean raw value and colors rank the mean absolute difference from ground truth across matched sequences. Ground truth is a reference row. Count ratio shows the mean ratio and, in parentheses, the mean per-sequence absolute deviation from 1 used for ranking. A dash indicates unavailable data. See the report inventory to check coverage and the selected runs.

Tables combine saved runs with matching recorded scoring parameters for each metric. Duplicate sequence/method rows prefer a finite completed score, then the newest CSV. RUN_OVERRIDE pins a single run.


In [ ]:
from pathlib import Path
import json
from html import escape

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "benchmarks/run_metric.py").is_file()), Path.cwd())
REPORT_ROOT = root / "reports"
SPLIT = "test"
# Optional: choose a specific run directory name per (dataset, metric, block size).
# Use None for the block size of a non-voxel metric.
RUN_OVERRIDE = {}

In [ ]:
MAIN = {
    "chamfer_full": "Chamfer Full ↓", "iei_distance": "IEI distance ↓",
    "fano_factor": "Fano error ↓", "event_count_ratio": "Count ratio ≈1",
    "event_count": "Event count", "event_rate": "Event rate (events/s)",
    "on_percentage": "ON (%)", "isi_median_ms": "Median ISI (ms)",
    "isi_cv": "ISI CV", "same_ts": "Same-ts fraction",
}
VOXEL = {"voxel_mae": "MAE", "voxel_rmse": "RMSE"}
CHANNELS = {"value": "Signed", "value_on": "ON", "value_off": "OFF"}
LOWER = {"chamfer_full", "iei_distance", "fano_factor", *VOXEL}
runs = {}
for cfg_path in sorted(REPORT_ROOT.glob("*/*/*/config.json")):
    cfg = json.loads(cfg_path.read_text())
    metric = cfg["metric"]
    if cfg["split"] != SPLIT or metric not in {*MAIN, *VOXEL}:
        continue
    size = cfg.get("options", {}).get("block_size", 1) if metric in VOXEL else None
    csv = cfg_path.with_name("results.csv")
    if csv.is_file():
        runs.setdefault((cfg["dataset"], metric, size), []).append(csv)

reports, inventory = {}, []
for key, paths in runs.items():
    override = RUN_OVERRIDE.get(key)
    choices = [p for p in paths if override is None or p.parent.name == override]
    if not choices:
        raise ValueError(f"Unknown run {override!r} for {key}")
    choices.sort(key=lambda p: (p.stat().st_mtime_ns, str(p)))
    def protocol(path):
        cfg = json.loads(path.with_name("config.json").read_text())
        opts = dict(cfg["options"])
        if key[1] == "chamfer_full":
            for field in ("device", "workers", "chunk_size", "backend_version"):
                opts.pop(field, None)
        return cfg["unit"], cfg["interval"], cfg["crop"], opts

    anchor = protocol(choices[-1])
    frames, sources = [], []
    for path in choices:
        if protocol(path) != anchor:
            print(f"Skipped different protocol: {path.parent}")
            continue
        frame = pd.read_csv(path)
        if frame.duplicated(["sequence", "method"]).any():
            raise ValueError(f"Duplicate sequence/method rows in {path}")
        sources.append(path.parent.name)
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    # Prefer a completed score, then the latest CSV for the same sequence/method.
    usable = raw.status.eq("ok") | ((key[1] == "fano_factor") & raw.status.eq("partial_support"))
    raw["_usable"] = usable & np.isfinite(pd.to_numeric(raw.value, errors="coerce"))
    raw = (raw.sort_values("_usable", kind="stable")
              .drop_duplicates(["sequence", "method"], keep="last").drop(columns="_usable"))
    ok = raw.status.eq("ok") | ((key[1] == "fano_factor") & raw.status.eq("partial_support"))
    rows = raw.copy()
    for field in CHANNELS:
        rows[field] = pd.to_numeric(rows.get(field, np.nan), errors="coerce")
        rows.loc[~ok | ~np.isfinite(rows[field]), field] = np.nan
    reports[key] = rows
    inventory.append(dict(dataset=key[0], metric=key[1], block=key[2], run=', '.join(sources),
                          candidates=len(paths), scored=int(rows.value.notna().sum()), rows=len(raw),
                          partial_support=int(raw.status.eq("partial_support").sum())))
if not reports:
    raise FileNotFoundError(f"No matching {SPLIT} reports under {REPORT_ROOT}")
if set(RUN_OVERRIDE) - set(reports):
    raise ValueError(f"Missing report overrides: {set(RUN_OVERRIDE) - set(reports)}")
datasets = sorted({key[0] for key in reports})
voxel_sizes = sorted({size for _, metric, size in reports if metric in VOXEL})
print(f"{SPLIT} | {', '.join(datasets)} | {len(reports)} selected reports")
with_inventory = widgets.Output()
with with_inventory:
    print("Compatible runs are combined per metric. Use RUN_OVERRIDE to pin a single run.")
    display(pd.DataFrame(inventory).fillna("").sort_values(["dataset", "metric"]))
details = widgets.Accordion(children=[with_inventory], selected_index=None)
details.set_title(0, "Selected reports and coverage")
display(details)

In [ ]:
def summarize(dataset, metric, field, sizes):
    matrices, common = {}, None
    for size in sizes:
        rows = reports.get((dataset, metric, size))
        if rows is None:
            continue
        matrix = rows.pivot(index="sequence", columns="method", values=field)
        matrices[size] = matrix
        available = matrix.drop(columns="ground_truth", errors="ignore").dropna(axis=1, how="all")
        if available.shape[1]:
            complete = available.dropna().index
            common = complete if common is None else common.intersection(complete)
    columns = {}
    for size, matrix in matrices.items():
        paired = matrix.reindex([] if common is None else common)
        descriptor = metric in MAIN and metric not in LOWER and metric != "event_count_ratio"
        if descriptor:
            paired = paired.loc[paired["ground_truth"].notna()] if "ground_truth" in paired else paired.iloc[:0]
        means = paired.mean()
        ranks = pd.Series(dtype=float)
        deviation = None
        if metric == "event_count_ratio":
            deviation = paired.sub(1).abs().mean()
            ranks = deviation
        elif metric in LOWER:
            ranks = means
        elif descriptor and "ground_truth" in paired:
            ranks = paired.sub(paired["ground_truth"], axis="index").abs().mean()
        ranks = ranks.drop(labels="ground_truth", errors="ignore").rank(method="dense")
        columns[size] = (means, ranks, deviation, len(paired))
    return columns

# Cache summaries so toggles only redraw tables.
tables = {}
for dataset in datasets:
    main, voxel = {}, {}
    for metric, label in MAIN.items():
        main[label] = summarize(dataset, metric, "value", [None]).get(None)
    for metric, short in VOXEL.items():
        for field, channel in CHANNELS.items():
            by_size = summarize(dataset, metric, field, voxel_sizes)
            for size in voxel_sizes:
                voxel[f"{short} {channel} {size}×{size}"] = by_size.get(size)
    tables[dataset] = main, voxel

In [ ]:
def controls(labels, defaults):
    return {label: widgets.Checkbox(value=label in defaults, description=label, indent=False,
                                    layout=widgets.Layout(width="175px")) for label in labels}

def render(title, columns, selected, methods):
    labels = [label for label in columns if label in selected]
    if not labels:
        return
    palette = ["#164b80", "#75a9d6", "#b4d4ed", "#e0eef8"]
    html = [f"<h4>{escape(title)}</h4><div style='overflow-x:auto'><table><thead><tr><th>Method</th>"]
    for label in labels:
        col = columns[label]
        count = col[3] if col else 0
        html.append(f"<th>{escape(label)}<br><small>n={count}</small></th>")
    html.append("</tr></thead><tbody>")
    for method in methods:
        if method == "ground_truth" and not any(columns[l] and pd.notna(columns[l][0].get(method)) for l in labels):
            continue
        html.append(f"<tr><th>{escape(str(method))}</th>")
        for label in labels:
            col = columns[label]
            value = col[0].get(method, np.nan) if col else np.nan
            rank = col[1].get(method, np.nan) if col else np.nan
            style, number = "", "—"
            if pd.notna(value):
                number = f"{value:,.5g}"
                if col[2] is not None:
                    number += f" ({col[2][method]:.4g})"
                if pd.notna(rank) and rank <= 4:
                    color = "white" if rank == 1 else "#102b43"
                    style = f"background:{palette[int(rank)-1]};color:{color}"
            html.append(f"<td style='{style}'>{number}</td>")
        html.append("</tr>")
    display(HTML("".join(html) + "</tbody></table></div>"))

first_main, first_voxel = tables[datasets[0]]
main_checks = controls(first_main, [label for label in first_main
                                    if any(tables[d][0][label] is not None for d in datasets)])
voxel_checks = controls(first_voxel, [label for label in first_voxel if "Signed" in label])
for title, checks in [("Main table columns", main_checks), ("Voxel comparison columns", voxel_checks)]:
    display(widgets.Label(title), widgets.Box(list(checks.values()), layout=widgets.Layout(flex_flow="row wrap")))
def redraw(**enabled):
    selected = {label for label, checked in enabled.items() if checked}
    for dataset, (main, voxel) in tables.items():
        methods = sorted({m for key, rows in reports.items() if key[0] == dataset for m in rows.method.unique()})
        render(f"{dataset} · Main results", main, selected, methods)
        if any(col is not None for col in voxel.values()):
            render(f"{dataset} · Voxel errors by spatial scale", voxel, selected, methods)
        else:
            print(f"{dataset}: no voxel MAE/RMSE reports available.")
display(widgets.interactive_output(redraw, {**main_checks, **voxel_checks}))